<a href="https://colab.research.google.com/github/Gaurav-0911/-Feb-Internship-NLP-Preprocessing-Engine/blob/main/Task_2_NLP_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Data Understanding


In [11]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [12]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [13]:
df.columns

Index(['review', 'sentiment'], dtype='object')

#2. NLP Preprocessing

In [14]:
# Import required libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords (run once)
nltk.download('stopwords')

# Initialize stopwords and stemmer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Function to preprocess text
def preprocess_text(text):
    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Remove special characters, numbers, and punctuation
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # 3. Tokenize text into words
    words = text.split()

    # 4. Remove stopwords (common words like 'the', 'is', etc.)
    words = [word for word in words if word not in stop_words]

    # 5. Apply stemming (reduce words to root form)
    words = [stemmer.stem(word) for word in words]

    # 6. Join words back into a single string
    return " ".join(words)

# Apply preprocessing function to the 'review' column
df['clean_text'] = df['review'].apply(preprocess_text)

# Display original and cleaned text
df[['review', 'clean_text']].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


#3. Feature Engineering

In [16]:
# Import required libraries
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# =========================
# 1. Bag of Words (BoW)
# =========================

# Initialize CountVectorizer (you can limit features for performance)
bow = CountVectorizer(max_features=5000)

# Fit and transform the clean text data
X_bow = bow.fit_transform(df['clean_text'])

# Convert to array (optional - for viewing)
X_bow_array = X_bow.toarray()

print("BoW Shape:", X_bow_array.shape)


# =========================
# 2. TF-IDF
# =========================

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Fit and transform the clean text data
X_tfidf = tfidf.fit_transform(df['clean_text'])

# Convert to array (optional)
X_tfidf_array = X_tfidf.toarray()

print("TF-IDF Shape:", X_tfidf_array.shape)

BoW Shape: (50000, 5000)
TF-IDF Shape: (50000, 5000)


In [18]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 16.4 MB/s eta 0:00:00


In [19]:
# Import Word2Vec
from gensim.models import Word2Vec

# Tokenize sentences (Word2Vec needs list of words)
tokenized_text = [text.split() for text in df['clean_text']]

# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=tokenized_text,
    vector_size=100,   # size of word vectors
    window=5,          # context window
    min_count=2,       # ignore rare words
    workers=4
)

# Function to get average Word2Vec for a sentence
import numpy as np

def get_avg_word2vec(words, model, vector_size):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(vector_size)

    return np.mean(vectors, axis=0)

# Create feature vectors for all sentences
X_w2v = np.array([
    get_avg_word2vec(text.split(), w2v_model, 100)
    for text in df['clean_text']
])

print("Word2Vec Shape:", X_w2v.shape)

Word2Vec Shape: (50000, 100)


#4. Model Building

In [20]:
# Import required libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# 1. Define Features (X) and Target (y)
# =========================

# Use TF-IDF features (recommended)
X = X_tfidf   # you can also use X_bow

# Convert sentiment to numeric (positive=1, negative=0)
y = df['sentiment'].map({'positive': 1, 'negative': 0})

# =========================
# 2. Train-Test Split
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 3. Logistic Regression
# =========================

lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("🔹 Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


# =========================
# 4. Naive Bayes
# =========================

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("🔹 Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


# =========================
# 5. Decision Tree
# =========================

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("🔹 Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


# =========================
# 6. Random Forest (Optional)
# =========================

rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("🔹 Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

🔹 Logistic Regression Accuracy: 0.8862
              precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

🔹 Naive Bayes Accuracy: 0.8516
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      4961
           1       0.85      0.86      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

🔹 Decision Tree Accuracy: 0.7127
              precision    recall  f1-score   support

           0       0.70      0.72      0.71      4961
           1       0.72      0.70      0.71      5039

    accuracy                           0.71     10000
   macro avg       0.71  

In [21]:
!pip install xgboost

In [22]:
from xgboost import XGBClassifier

# Initialize model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Train model
xgb_model.fit(X_train, y_train)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)

# Evaluation
print("🔹 XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:37:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🔹 XGBoost Accuracy: 0.8553
              precision    recall  f1-score   support

           0       0.87      0.84      0.85      4961
           1       0.84      0.88      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



#5. Model Evaluation

In [23]:
# Import required metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# =========================
# 1. Create Evaluation Function
# =========================

def evaluate_model(y_test, y_pred, model_name):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }

# =========================
# 2. Evaluate All Models
# =========================

results = []

results.append(evaluate_model(y_test, y_pred_lr, "Logistic Regression"))
results.append(evaluate_model(y_test, y_pred_nb, "Naive Bayes"))
results.append(evaluate_model(y_test, y_pred_dt, "Decision Tree"))
results.append(evaluate_model(y_test, y_pred_rf, "Random Forest"))

# Optional (if you used XGBoost)
try:
    results.append(evaluate_model(y_test, y_pred_xgb, "XGBoost"))
except:
    pass

# =========================
# 3. Create Comparison Table
# =========================

results_df = pd.DataFrame(results)

# Sort by F1 Score (best model on top)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

# Display results
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8862,0.876181,0.901568,0.888693
4,XGBoost,0.8553,0.843535,0.875174,0.859063
3,Random Forest,0.8544,0.864200,0.843620,0.853786
1,Naive Bayes,0.8516,0.848325,0.859099,0.853678
2,Decision Tree,0.7127,0.721201,0.700734,0.710820


# 6. Comparison & Insights

## Best Preprocessing
The preprocessing steps included lowercasing, removal of special characters, stopword removal, and stemming.  
These steps helped in reducing noise and improving the quality of text data.

Final: Cleaning + Stopword Removal + Stemming gave the best results.

---

## Best Vectorization
Bag of Words (BoW) was simple but did not consider word importance.  
TF-IDF performed better as it assigns importance to words based on frequency.

Final: TF-IDF is better than BoW.

---

## Best Model
Logistic Regression and Naive Bayes gave the best performance.  
Decision Tree showed overfitting, while Random Forest was more stable but slower.

Final: TF-IDF + Logistic Regression / Naive Bayes is the best combination.

---

## Trade-offs
- Higher accuracy models require more computation  
- Simpler models are faster but less powerful  
- Complex models are more accurate but slower  

Final: A balance between accuracy, speed, and complexity is important.

---

# Pipeline Flow
Raw Data → Preprocessing → Feature Engineering → Model Training → Evaluation → Comparison

---

# Final Summary
TF-IDF with Logistic Regression or Naive Bayes provides the best balance of performance and efficiency.  
Preprocessing plays a crucial role in improving model accuracy.